In [2]:
# =============================================================================
# LOAD HPO BEST PARAMETERS
# =============================================================================

import json
from pathlib import Path
import numpy as np

# Load from HPO
HPO_FILE = Path("../../runs/hpo/best_parameters.json")
with open(HPO_FILE) as f:
    hpo_data = json.load(f)

best_params = hpo_data["best_params"]
hpo_val_acc = hpo_data["best_accuracy"]

print("=" * 80)
print("HPO OPTIMIZED PARAMETERS")
print("=" * 80)
print(f"HPO validation accuracy: {hpo_val_acc:.4f}")
print("\nParameters:")
for k, v in best_params.items():
    print(f"  {k:20s}: {v}")
print("=" * 80)

# Extract parameters
CHANNELS_MULT = best_params["channels_multiplier"]
DROPOUT = best_params["dropout"]
L2_REG = best_params["l2_reg"]
LEARNING_RATE = best_params["learning_rate"]
BATCH_SIZE = best_params["batch_size"]
AUGMENTATION = best_params["augmentation"]

# Architecture
BASE_CHANNELS = (64, 128, 512, 512, 128)
CHANNELS = tuple(int(c * CHANNELS_MULT) for c in BASE_CHANNELS)
CONVS_PER_BLOCK = 2

# Training
EXPERIMENT_NAME = "hpo_test"
MODEL_NAME = "hpo_optimized"
EPOCHS = 50
PATIENCE = 10

print(f"\nFinal channels: {CHANNELS}")
print(f"Training: {EPOCHS} epochs, patience={PATIENCE}")


HPO OPTIMIZED PARAMETERS
HPO validation accuracy: 0.7677

Parameters:
  architecture        : complex
  channels_multiplier : 1.2
  dropout             : 0.4274714954903845
  l2_reg              : 0.0013179985689176007
  learning_rate       : 0.0001001283474556078
  batch_size          : 64
  augmentation        : medium

Final channels: (76, 153, 614, 614, 153)
Training: 50 epochs, patience=10


In [3]:
# =============================================================================
# IMPORTS
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import keras
from keras import layers
from keras.utils import image_dataset_from_directory
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import time
from datetime import datetime

# GPU config
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Constants
CLASS_ORDER = ['happy', 'neutral', 'sad', 'surprise']
CLASS_WEIGHTS = {0: 0.950, 1: 0.950, 2: 0.949, 3: 1.190}
NUM_CLASSES = 4
IMG_SIZE = (48, 48)
METRIC_NAME = "sparse_categorical_accuracy"
VAL_METRIC_NAME = f"val_{METRIC_NAME}"

TRAIN_DIR = Path("../../data/train")
VAL_DIR = Path("../../data/validation")
TEST_DIR = Path("../../data/test")
MODELS_DIR = Path("../../models")

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {gpus}")


TensorFlow: 2.20.0
GPU: []


In [4]:
# =============================================================================
# DATA LOADERS
# =============================================================================

def make_generators():
    def preprocess(image, label):
        return tf.cast(image, tf.float32) / 255.0, label
    
    def augment_medium(image, label):
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.2)
        image = tf.image.random_contrast(image, 0.8, 1.2)
        return tf.clip_by_value(image, 0.0, 1.0), label
    
    train_ds = image_dataset_from_directory(
        TRAIN_DIR, class_names=CLASS_ORDER, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, color_mode="rgb", shuffle=True
    ).map(preprocess).map(augment_medium if AUGMENTATION=="medium" else preprocess)
    
    val_ds = image_dataset_from_directory(
        VAL_DIR, class_names=CLASS_ORDER, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, color_mode="rgb", shuffle=False
    ).map(preprocess)
    
    test_ds = image_dataset_from_directory(
        TEST_DIR, class_names=CLASS_ORDER, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, color_mode="rgb", shuffle=False
    ).map(preprocess)
    
    return train_ds, val_ds, test_ds

train_ds, val_ds, test_ds = make_generators()
print("✓ Data loaded")


Found 15109 files belonging to 4 classes.
Found 4977 files belonging to 4 classes.
Found 128 files belonging to 4 classes.
✓ Data loaded


In [5]:
# =============================================================================
# BUILD MODEL WITH HPO PARAMETERS
# =============================================================================

def build_model():
    input_shape = IMG_SIZE + (3,)
    inputs = keras.Input(shape=input_shape)
    x = inputs
    
    for i, filters in enumerate(CHANNELS):
        for j in range(CONVS_PER_BLOCK):
            x = layers.Conv2D(filters, 3, padding="same",
                            kernel_regularizer=keras.regularizers.l2(L2_REG))(x)
            x = layers.BatchNormalization()(x)
            x = layers.LeakyReLU()(x)
        if i < len(CHANNELS) - 1:
            x = layers.MaxPooling2D(2)(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=[METRIC_NAME]
    )
    return model

model = build_model()
print(f"✓ Model built: {model.count_params():,} parameters")


✓ Model built: 12,466,614 parameters


In [6]:
# =============================================================================
# TRAIN AND EVALUATE
# =============================================================================

print("=" * 80)
print("TRAINING HPO-OPTIMIZED MODEL")
print("=" * 80)

callbacks = [
    keras.callbacks.EarlyStopping(monitor=VAL_METRIC_NAME, patience=PATIENCE, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor=VAL_METRIC_NAME, factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(str(MODELS_DIR / f"best_{MODEL_NAME}.keras"), monitor=VAL_METRIC_NAME, save_best_only=True, verbose=1)
]

start = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, class_weight=CLASS_WEIGHTS, verbose=1)
train_time = time.time() - start

print(f"\n✓ Training: {train_time/60:.1f} min")
print(f"Best val acc: {max(history.history[VAL_METRIC_NAME]):.4f}")

# Evaluate on test
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(preds.argmax(axis=1))

test_acc = f1_score(y_true, y_pred, average='weighted')

print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(f"HPO validation:  {hpo_val_acc:.1%}")
print(f"Final val:       {max(history.history[VAL_METRIC_NAME]):.1%}")
print(f"TEST ACCURACY:   {test_acc:.1%}")
print(f"\nComparison:")
print(f"  Baseline:      80.0%")
print(f"  HPO optimized: {test_acc:.1%}")
print(f"  Difference:    {(test_acc - 0.800)*100:+.1f}%")
print("=" * 80)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_ORDER))


TRAINING HPO-OPTIMIZED MODEL
Epoch 1/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 621ms/step - loss: 4.9174 - sparse_categorical_accuracy: 0.4492
Epoch 1: val_sparse_categorical_accuracy improved from None to 0.23086, saving model to ..\..\models\best_hpo_optimized.keras
237/237 ━━━━━━━━━━━━━━━━━━━━ 169s 691ms/step - loss: 4.6585 - sparse_categorical_accuracy: 0.5368 - val_loss: 5.0620 - val_sparse_categorical_accuracy: 0.2309 - learning_rate: 1.0013e-04
Epoch 2/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 568ms/step - loss: 4.1767 - sparse_categorical_accuracy: 0.6472
Epoch 2: val_sparse_categorical_accuracy improved from 0.23086 to 0.67008, saving model to ..\..\models\best_hpo_optimized.keras
237/237 ━━━━━━━━━━━━━━━━━━━━ 149s 629ms/step - loss: 4.0430 - sparse_categorical_accuracy: 0.6658 - val_loss: 3.8823 - val_sparse_categorical_accuracy: 0.6701 - learning_rate: 1.0013e-04
Epoch 3/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 597ms/step - loss: 3.7012 - sparse_categorical_accuracy: 0.7008
Epoch 3: val_sparse_c